In [1]:
#Already done, output of this cell already available in dataset_big
# https://edgar.jrc.ec.europa.eu/dataset_ap81#p3
#Script to average two annual NetCDF files for NOx and two for CO, clip to Sub-Saharan Africa, and output two GeoTIFF files.
"""
import numpy as np
import xarray as xr
import rioxarray

def main():
    # Input files with absolute paths
    nox1 = r"C:/Users/matti/Desktop/nox_21.nc"
    nox2 = r"C:/Users/matti/Desktop/nox_22.nc"
    co1 = r"C:/Users/matti/Desktop/co_21.nc"
    co2 = r"C:/Users/matti/Desktop/co_22.nc"
    
    # Define bounding box for Sub-Saharan Africa
    lon_min, lon_max = -20, 55
    lat_min, lat_max = -35, 20
    
    # Process NOx
    nox_files = [nox1, nox2]
    nox_da = average_pollutant(nox_files, lon_min, lon_max, lat_min, lat_max)
    # Process CO
    co_files = [co1, co2]
    co_da = average_pollutant(co_files, lon_min, lon_max, lat_min, lat_max)
    
    # Write GeoTIFFs to desktop
    nox_da.rio.to_raster("dataset_big/NOx_SSA_avg.tif")
    co_da.rio.to_raster("dataset_big/CO_SSA_avg.tif")
    print("Output files written: NOx_SSA_avg.tif, CO_SSA_avg.tif")

def average_pollutant(files, lon_min, lon_max, lat_min, lat_max):
    #Read two NetCDF files, clip, average pixel-wise, return xarray DataArray.
    datasets = []
    for f in files:
        ds = xr.open_dataset(f)
        # Identify the data variable (first variable with lat/lon dims)
        data_var = None
        for var in ds.data_vars:
            if 'lat' in ds[var].dims and 'lon' in ds[var].dims:
                data_var = var
                break
        if data_var is None:
            raise ValueError(f"No variable with lat/lon dimensions in {f}")
        da = ds[data_var]
        # If there's a time dimension, average over it (assume annual file may have time)
        if 'time' in da.dims:
            da = da.mean(dim='time', keep_attrs=True)
        # Assign CRS (WGS84) for rioxarray
        da = da.rio.write_crs("EPSG:4326")
        # Clip to bounding box
        da = da.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
        datasets.append(da)
    # Pixel-wise average of the two files
    avg_da = (datasets[0] + datasets[1]) / 2.0
    # Preserve attributes
    avg_da.attrs = datasets[0].attrs
    return avg_da

if __name__ == "__main__":
    main()


"""

'\nimport numpy as np\nimport xarray as xr\nimport rioxarray\n\ndef main():\n    # Input files with absolute paths\n    nox1 = r"C:/Users/matti/Desktop/nox_21.nc"\n    nox2 = r"C:/Users/matti/Desktop/nox_22.nc"\n    co1 = r"C:/Users/matti/Desktop/co_21.nc"\n    co2 = r"C:/Users/matti/Desktop/co_22.nc"\n\n    # Define bounding box for Sub-Saharan Africa\n    lon_min, lon_max = -20, 55\n    lat_min, lat_max = -35, 20\n\n    # Process NOx\n    nox_files = [nox1, nox2]\n    nox_da = average_pollutant(nox_files, lon_min, lon_max, lat_min, lat_max)\n    # Process CO\n    co_files = [co1, co2]\n    co_da = average_pollutant(co_files, lon_min, lon_max, lat_min, lat_max)\n\n    # Write GeoTIFFs to desktop\n    nox_da.rio.to_raster("dataset_big/NOx_SSA_avg.tif")\n    co_da.rio.to_raster("dataset_big/CO_SSA_avg.tif")\n    print("Output files written: NOx_SSA_avg.tif, CO_SSA_avg.tif")\n\ndef average_pollutant(files, lon_min, lon_max, lat_min, lat_max):\n    #Read two NetCDF files, clip, average pi

In [2]:
import rioxarray
import numpy as np

nox = rioxarray.open_rasterio("dataset_big/NOx_SSA_avg.tif").values.flatten()
co = rioxarray.open_rasterio("dataset_big/CO_SSA_avg.tif").values.flatten()

mask = np.isfinite(nox) & np.isfinite(co)
corr = np.corrcoef(nox[mask], co[mask])[0,1]
print(f"correlation NOx-CO: {corr:.4f}")

correlation NOx-CO: 0.9712


In [3]:
"""
Script to compute weighted average of NOx.  we esclude CO as correlation is very high--we put co weight=0.
Reads from dataset_big/NOx_SSA_avg.tif [and dataset_big/CO_SSA_avg.tif],
applies user-defined weights, and writes dataset_big/vehicles_emission_SSA.tif.
"""

import numpy as np
import rioxarray

def main():

    normalize = True          # to asses for difference in scale between NOx and CO
    norm_method = "minmax"    # option: "minmax", "zscore", "none"

    # User can change these weights at the top of the script
    weight_nox = 1
    weight_co = 0

    # Input file paths (relative to current working directory)
    nox_path = "dataset_big/NOx_SSA_avg.tif"
    co_path = "dataset_big/CO_SSA_avg.tif"
    output_path = "dataset_big/vehicles_emission_SSA.tif"

    # Read the GeoTIFFs with rioxarray
    nox_da = rioxarray.open_rasterio(nox_path)
    co_da = rioxarray.open_rasterio(co_path)

    #normalize
    if normalize:
        if norm_method == "minmax":
            nox_min = nox_da.min().item()
            nox_max = nox_da.max().item()
            co_min = co_da.min().item()
            co_max = co_da.max().item()
            nox_da = (nox_da - nox_min) / (nox_max - nox_min)
            co_da = (co_da - co_min) / (co_max - co_min)
        elif norm_method == "zscore":
            nox_mean = nox_da.mean().item()
            nox_std = nox_da.std().item()
            co_mean = co_da.mean().item()
            co_std = co_da.std().item()
            nox_da = (nox_da - nox_mean) / nox_std
            co_da = (co_da - co_mean) / co_std
        elif norm_method == "none":
            pass
        else:
            raise ValueError(f"Unknown normalization method: {norm_method}")

    # Compute weighted average
    weighted_avg = (weight_nox * nox_da) + (weight_co * co_da)

    # Preserve metadata (optional)
    weighted_avg.attrs = nox_da.attrs
    weighted_avg.rio.write_crs(nox_da.rio.crs, inplace=True)

    # Write output GeoTIFF
    weighted_avg.rio.to_raster(output_path)

    print(f"Output written to {output_path}")
    print(f"Weights: NOx = {weight_nox}, CO = {weight_co}")

if __name__ == "__main__":
    main()

Output written to dataset_big/vehicles_emission_SSA.tif
Weights: NOx = 1, CO = 0


In [4]:
"""
Script to clip the vehicle emission raster to Nigeria, reproject to Web Mercator (EPSG:3857),
resample to 1 km resolution, and rescale values to [0, 1] inside Nigeria.
"""

import geopandas as gpd
import rioxarray
import numpy as np
from shapely import unary_union
import xarray as xr
from shapely.geometry import mapping
from rasterio.warp import Resampling

def main():
    
    # User parameters
    raster_path = "dataset_big/vehicles_emission_SSA.tif"
    geojson_path = "dataset_big/Country_boundaries.geojson"
    output_path = "dataset_big/emission_nigeria_3857.tif"
    target_resolution = 1000  # meters in EPSG:3857 (1 km)
    resample_method = Resampling.nearest
    target_crs = "EPSG:3857"
    rescale_to_unit = True
    output_nodata = -9999.0
    
    # 1. Read the raster
    print("Reading emission raster...")
    da = rioxarray.open_rasterio(raster_path)
    original_crs = da.rio.crs
    print(f"Original CRS: {original_crs}")
    print(f"Original resolution: {da.rio.resolution()}")
    
    # 2. Read the GeoJSON and extract Nigeria polygon
    print(f"Reading GeoJSON from {geojson_path}...")
    gdf = gpd.read_file(geojson_path)
    print(f"GeoJSON columns: {gdf.columns.tolist()}")
    
    nigeria_geom = unary_union(gdf.geometry)
    nigeria_geom_3857 = gpd.GeoSeries([nigeria_geom], crs=gdf.crs).to_crs(target_crs).iloc[0]
    
    # 3. Reproject raster to target CRS (EPSG:3857) and clip to Nigeria
    print(f"Reprojecting raster to {target_crs}...")
    da_3857 = da.rio.reproject(target_crs)
    
    print("Clipping raster to Nigeria geometry...")
    da_clipped = da_3857.rio.clip([mapping(nigeria_geom_3857)], drop=True)
    
    # 4. Resample to 1 km resolution
    print(f"Resampling to {target_resolution} m resolution using method '{resample_method}'...")
    da_resampled = da_clipped.rio.reproject(
        target_crs,
        resolution=target_resolution,
        resampling=resample_method,
        nodata=output_nodata
    )

    # Keep the Nigeria mask after resampling and enforce nodata outside border
    da_resampled = da_resampled.rio.clip([mapping(nigeria_geom_3857)], drop=False)
    da_resampled = da_resampled.fillna(output_nodata)
    da_resampled.rio.write_nodata(output_nodata, inplace=True)

    # 5. Rescale to [0, 1] on valid Nigeria pixels only
    if rescale_to_unit:
        print("Rescaling values to [0, 1] within Nigeria...")
        data = da_resampled.values.astype(np.float32, copy=True)
        valid = np.isfinite(data) & (data != output_nodata)

        if valid.any():
            vmin = data[valid].min()
            vmax = data[valid].max()

            if vmax > vmin:
                data[valid] = (data[valid] - vmin) / (vmax - vmin)
            else:
                data[valid] = 0.0

            data[~valid] = output_nodata

            da_resampled = xr.DataArray(
                data,
                coords=da_resampled.coords,
                dims=da_resampled.dims,
                attrs=da_resampled.attrs,
                name=da_resampled.name,
            )
            da_resampled.rio.write_crs(target_crs, inplace=True)
            da_resampled.rio.write_nodata(output_nodata, inplace=True)
            print(f"Rescaled min={float(data[valid].min()):.4f}, max={float(data[valid].max()):.4f}")
            print(f"Nodata value={output_nodata}, nodata pixels={int((~valid).sum())}")
        else:
            print("No valid pixels found for rescaling.")
    
    # 6. Write output
    print(f"Writing output to {output_path}...")
    da_resampled.rio.to_raster(output_path, compress="DEFLATE")
    print("Done.")
    
if __name__ == "__main__":
    main()

Reading emission raster...
Original CRS: EPSG:4326
Original resolution: (0.1, 0.1)
Reading GeoJSON from dataset_big/Country_boundaries.geojson...
GeoJSON columns: ['fid', 'GID_0', 'NAME_0', 'GID_1', 'NAME_1', 'VARNAME_1', 'NL_NAME_1', 'TYPE_1', 'ENGTYPE_1', 'CC_1', 'HASC_1', 'geometry']
Reprojecting raster to EPSG:3857...
Clipping raster to Nigeria geometry...
Resampling to 1000 m resolution using method '0'...
Rescaling values to [0, 1] within Nigeria...
Rescaled min=0.0000, max=1.0000
Nodata value=-9999.0, nodata pixels=510628
Writing output to dataset_big/emission_nigeria_3857.tif...
Done.


In [5]:
"""
Create an income raster for Nigeria using OnStove income_estimation.
This script uses only method 1 (AWE estimation) in this notebook.
Method 2 (Income CSV interpolation) is intentionally commented out.
"""

import os
import shutil
from onstove import OnStove

# User parameters (paths relative to this notebook folder)
model_pickle_path = os.path.normpath("../../example/NGA/model_inputs.pkl")
scenario_csv_path = os.path.normpath("../../example/NGA/soc_specs.csv")
output_directory = "dataset_big"

# Method 2 not used in this notebook (kept as reference)
# use_income_csv = True
# income_csv_path = "dataset_big/nigeria_income_percentiles.csv"

# Required for AWE mode
gini_value = 0.339      # Example: 0.35
gdp_pc_value = 2139    # Example: 2200.0
pareto_weight = 0.32

# Output raster column and name (AWE writes absolute_wealth)
output_variable = "absolute_wealth"
output_raster_name = "income_nigeria"

# Basic input checks
if not os.path.exists(model_pickle_path):
    raise FileNotFoundError(f"Model pickle not found: {model_pickle_path}")
if not os.path.exists(scenario_csv_path):
    raise FileNotFoundError(f"Scenario csv not found: {scenario_csv_path}")
os.makedirs(output_directory, exist_ok=True)

# 1) Load prepared model and scenario
model = OnStove.read_model(model_pickle_path)
model.read_scenario_data(scenario_csv_path, delimiter=",")
model.output_directory = output_directory

# 2) Configure inputs for selected income mode
# AWE only branch
if gini_value is None or gdp_pc_value is None:
    raise ValueError("Set gini_value and gdp_pc_value for AWE mode")
model.specs["gini"] = float(gini_value)
model.specs["gdp_pc"] = float(gdp_pc_value)
model.income_estimation(awe=True, income_data=None, pareto_weight=pareto_weight)

# 3) Quick checks
if "relative_wealth" not in model.gdf.columns:
    raise KeyError("Model gdf is missing 'relative_wealth', required by income_estimation")
if output_variable not in model.gdf.columns:
    raise KeyError(f"Expected output column '{output_variable}' not found in model.gdf")

# 4) Export raster (suppresses OnStove's internal print statements)
import contextlib
import io
with contextlib.redirect_stdout(io.StringIO()):
    model.to_raster(variable=output_variable)

# Move file to final destination and clean up
src_name = f"{output_variable}_mean.tif"
src_path = os.path.join(output_directory, "Rasters", src_name)
dst_path = os.path.join(output_directory, f"{output_raster_name}.tif")
if os.path.exists(src_path):
    if os.path.exists(dst_path):
        os.remove(dst_path)
    os.replace(src_path, dst_path)
    print(f"Raster moved to: {dst_path}")

# Delete the temporary Rasters directory
rasters_dir = os.path.join(output_directory, "Rasters")
if os.path.exists(rasters_dir):
    shutil.rmtree(rasters_dir)
    print(f"Temporary directory deleted: {rasters_dir}")

print("Income mode: AWE")
print(f"Final raster: {dst_path if os.path.exists(dst_path) else 'NOT FOUND'}")


Raster moved to: dataset_big\income_nigeria.tif
Temporary directory deleted: dataset_big\Rasters
Income mode: AWE
Final raster: dataset_big\income_nigeria.tif


In [6]:
"""
Reproject income raster from EPSG:3395 (or EPSG:4326) to EPSG:3857 (Web Mercator).
Normalize income to [0, 1], clip to Nigeria with nodata outside, ensuring alignment with emission raster.
"""

import rioxarray
import geopandas as gpd
import numpy as np
import xarray as xr
from rasterio.warp import Resampling
from shapely import unary_union
from shapely.geometry import mapping

def main():
    
    # User parameters
    input_path = "dataset_big/income_nigeria.tif"
    output_path = "dataset_big/income_nigeria_3857.tif"
    geojson_path = "dataset_big/Country_boundaries.geojson"
    target_crs = "EPSG:3857"
    target_resolution = 1000  # meters in EPSG:3857 (1 km), same as emission raster
    output_nodata = -9999.0
    
    # Read the income raster in its original CRS
    print(f"Reading income raster from {input_path}...")
    income_da = rioxarray.open_rasterio(input_path)
    original_crs = income_da.rio.crs
    original_nodata = income_da.rio.nodata
    print(f"Original CRS: {original_crs}")
    print(f"Original resolution: {income_da.rio.resolution()}")
    print(f"Original nodata value: {original_nodata}")
    print(f"Array contains NaN values: {np.isnan(income_da.values).any()}")
    
    # Reproject to target CRS (EPSG:3857) with same resolution as emission raster
    print(f"Reprojecting to {target_crs} with resolution {target_resolution} m...")
    income_3857 = income_da.rio.reproject(
        target_crs,
        resolution=target_resolution,
        resampling=Resampling.nearest
    )
    
    # Read Nigeria geometry and clip to it
    print(f"Reading GeoJSON from {geojson_path}...")
    gdf = gpd.read_file(geojson_path)
    nigeria_geom = unary_union(gdf.geometry)
    nigeria_geom_3857 = gpd.GeoSeries([nigeria_geom], crs=gdf.crs).to_crs(target_crs).iloc[0]
    
    print("Clipping income raster to Nigeria geometry...")
    income_3857 = income_3857.rio.clip([mapping(nigeria_geom_3857)], drop=False)
    income_3857 = income_3857.fillna(output_nodata)
    income_3857.rio.write_nodata(output_nodata, inplace=True)
    
    # Normalize income to [0, 1] within Nigeria only
    print("Normalizing income to [0, 1]...")
    data = income_3857.values.astype(np.float32, copy=True)
    # Identify valid pixels: exclude NaN, original_nodata (0.0), and output_nodata
    valid = (np.isfinite(data)) & (data != output_nodata)
    if original_nodata is not None:
        valid = valid & (data != original_nodata)
    
    if valid.any():
        vmin = data[valid].min()
        vmax = data[valid].max()
        
        if vmax > vmin:
            data[valid] = (data[valid] - vmin) / (vmax - vmin)
        else:
            data[valid] = 0.0
        
        # Set nodata pixels explicitly to output_nodata
        data[~valid] = output_nodata
        
        income_3857 = xr.DataArray(
            data,
            coords=income_3857.coords,
            dims=income_3857.dims,
            attrs=income_3857.attrs,
            name=income_3857.name,
        )
        income_3857.rio.write_crs(target_crs, inplace=True)
        income_3857.rio.write_nodata(output_nodata, inplace=True)
        print(f"Income normalized: min={float(data[valid].min()):.4f}, max={float(data[valid].max()):.4f}")
        print(f"Nodata value={output_nodata}, nodata pixels={int((~valid).sum())}")
    else:
        print("No valid pixels found for normalization.")
    
    # Write output with explicit nodata in metadata
    print(f"Writing normalized and reprojected raster to {output_path}...")
    income_3857.rio.to_raster(output_path, compress="DEFLATE", nodata=output_nodata)
    print(f"Done. Income raster saved as {output_path} in {target_crs}")
    
if __name__ == "__main__":
    main()



Reading income raster from dataset_big/income_nigeria.tif...
Original CRS: EPSG:3395
Original resolution: (1000.0, -1000.0)
Original nodata value: 0.0
Array contains NaN values: False
Reprojecting to EPSG:3857 with resolution 1000 m...
Reading GeoJSON from dataset_big/Country_boundaries.geojson...
Clipping income raster to Nigeria geometry...
Normalizing income to [0, 1]...
Income normalized: min=0.0000, max=1.0000
Nodata value=-9999.0, nodata pixels=803311
Writing normalized and reprojected raster to dataset_big/income_nigeria_3857.tif...
Done. Income raster saved as dataset_big/income_nigeria_3857.tif in EPSG:3857


In [7]:
"""
Script to compute weighted average of normalized income and vehicle emissions raster.
Reads from dataset_big/income_nigeria_3857.tif (already normalized to [0, 1]) 
and dataset_big/emission_nigeria_3857.tif (already normalized to [0, 1]),
applies weights (income=0.6, emissions=0.4), and writes dataset_big/vehicle_possibility.tif.
"""

import numpy as np
import rioxarray
import xarray as xr
from rasterio.warp import Resampling

def main():

    # Income is already normalized in the reprojection step, so no normalization needed here
    normalize_income = False
    
    # User defined weights
    weight_income = 0.6
    weight_emission = 0.4

    # Input file paths (relative to current working directory)
    income_path = "dataset_big/income_nigeria_3857.tif"
    emission_path = "dataset_big/emission_nigeria_3857.tif"
    output_path = "dataset_big/vehicle_possibility.tif"

    # Read the GeoTIFFs with rioxarray
    income_da = rioxarray.open_rasterio(income_path)
    emission_da = rioxarray.open_rasterio(emission_path)

    print(f"Income raster shape: {income_da.shape}, CRS: {income_da.rio.crs}")
    print(f"Emission raster shape: {emission_da.shape}, CRS: {emission_da.rio.crs}")

    # Clip both rasters to the intersection of their bounds (use emission as reference)
    print("Aligning rasters to same grid...")
    income_da = income_da.rio.clip_box(*emission_da.rio.bounds())
    
    # Resample income to match emission's exact grid
    income_da = income_da.rio.reproject_match(emission_da, resampling=Resampling.nearest)
    
    print(f"After alignment - Income shape: {income_da.shape}, Emission shape: {emission_da.shape}")

    # Income normalization is now done in the reprojection step, so we skip it here
    if normalize_income:
        print("Income is already normalized [0, 1] from reprojection step - skipping normalization")

    # Compute weighted average
    # Both rasters are already normalized to [0, 1]
    print(f"Computing weighted average with weights: Income={weight_income}, Emission={weight_emission}")
    weighted_avg = (weight_income * income_da) + (weight_emission * emission_da)

    # Set all negative valid pixels to 0 in the final raster
    output_nodata = emission_da.rio.nodata
    data = weighted_avg.values.astype(np.float32, copy=True)
    valid = np.isfinite(data)
    if output_nodata is not None:
        valid = valid & (data != output_nodata)
    negative_valid = valid & (data < 0)
    data[negative_valid] = 0.0

    weighted_avg = xr.DataArray(
        data,
        coords=weighted_avg.coords,
        dims=weighted_avg.dims,
        attrs=weighted_avg.attrs,
        name=weighted_avg.name,
    )

    # Preserve metadata from emission raster (already properly projected)
    weighted_avg.attrs = emission_da.attrs
    weighted_avg.rio.write_crs(emission_da.rio.crs, inplace=True)
    if output_nodata is not None:
        weighted_avg.rio.write_nodata(output_nodata, inplace=True)

    # Write output GeoTIFF
    weighted_avg.rio.to_raster(output_path, compress="DEFLATE")

    print(f"Negative valid pixels set to 0: {int(negative_valid.sum())}")
    print(f"Output written to {output_path}")
    print(f"Weights: Income = {weight_income}, Emission = {weight_emission}")
    print(f"Output range: min={float(weighted_avg.min()):.4f}, max={float(weighted_avg.max()):.4f}")

if __name__ == "__main__":
    main()


Income raster shape: (1, 1085, 1337), CRS: EPSG:3857
Emission raster shape: (1, 1089, 1327), CRS: EPSG:3857
Aligning rasters to same grid...
After alignment - Income shape: (1, 1089, 1327), Emission shape: (1, 1089, 1327)
Computing weighted average with weights: Income=0.6, Emission=0.4
Negative valid pixels set to 0: 295802
Output written to dataset_big/vehicle_possibility.tif
Weights: Income = 0.6, Emission = 0.4
Output range: min=-9999.0000, max=0.9984


In [8]:
"""
Vehicle allocation rationale

This cell distributes a fixed total number of vehicles across pixels in Nigeria,
combining:
1) vehicle_possibility (relative propensity to own a car)
2) population per pixel.

Method idea:
- the pixel with the highest vehicle_possibility is constrained to 100% adoption,
- all other pixels receive a lower adoption share, scaled smoothly,
- a calibration parameter (alpha) is found so that the sum of allocated cars
  is exactly equal to the national target (11,600,000). [OICA, 2020]

Final outputs are two separate rasters:
- vehicles_allocation_share.tif: share (%) of population owning a car in each pixel
- vehicles_allocation_n_effettivo.tif: number of cars allocated to each pixel
"""

import numpy as np
import xarray as xr
import rioxarray
from rasterio.warp import Resampling

# Inputs
vehicle_possibility_path = "dataset_big/vehicle_possibility.tif"
population_path = "dataset_big/Population.tif"
output_share_path = "dataset_big/vehicles_allocation_share.tif"
output_cars_path = "dataset_big/vehicles_allocation_n_effettivo.tif"

total_vehicles = 11_600_000
output_nodata = -9999.0

def total_cars_for_alpha(alpha, s_norm, pop):
    """Return total allocated cars for a given alpha."""
    rates = np.power(s_norm, alpha, dtype=np.float64)
    return float(np.sum(pop * rates, dtype=np.float64))

# 1) Read rasters and align population to the vehicle-possibility grid
score_da = rioxarray.open_rasterio(vehicle_possibility_path)
pop_da = rioxarray.open_rasterio(population_path)
pop_da = pop_da.rio.reproject_match(score_da, resampling=Resampling.nearest)

score_nodata = score_da.rio.nodata
pop_nodata = pop_da.rio.nodata

score = score_da.values.astype(np.float64, copy=False)
pop = pop_da.values.astype(np.float64, copy=False)

# 2) Build valid mask: finite, positive population, and non-nodata values
valid = np.isfinite(score) & np.isfinite(pop) & (pop > 0)
if score_nodata is not None:
    valid &= score != score_nodata
if pop_nodata is not None:
    valid &= pop != pop_nodata

if not valid.any():
    raise ValueError("No valid pixels found after masking.")

# 3) Convert vehicle possibility to [0, 1] among valid pixels
s = score[valid]
s_min = float(np.min(s))
s_max = float(np.max(s))

if s_max <= s_min:
    raise ValueError("Vehicle possibility has no variation on valid pixels.")

s_norm = (s - s_min) / (s_max - s_min)
s_norm = np.clip(s_norm, 0.0, 1.0)

# Force one max pixel to exactly 1.0 => 100% adoption in that pixel
max_idx = int(np.argmax(s))
s_norm[max_idx] = 1.0

pop_valid = pop[valid]

# 4) Calibrate alpha with bisection so sum(cars_i) = total_vehicles
cars_at_alpha0 = total_cars_for_alpha(0.0, s_norm, pop_valid)
if total_vehicles > cars_at_alpha0:
    raise ValueError(
        f"Requested total_vehicles={total_vehicles:,} exceeds population capacity={cars_at_alpha0:,.0f}."
    )

left, right = 0.0, 40.0
for _ in range(80):
    mid = 0.5 * (left + right)
    cars_mid = total_cars_for_alpha(mid, s_norm, pop_valid)
    if cars_mid > total_vehicles:
        left = mid
    else:
        right = mid

alpha = 0.5 * (left + right)

# 5) Compute final per-pixel outputs
rates_valid = np.power(s_norm, alpha, dtype=np.float64)  # 0..1
cars_valid = pop_valid * rates_valid

share_percent = np.full(score.shape, output_nodata, dtype=np.float32)
n_effettivo = np.full(score.shape, output_nodata, dtype=np.float32)

share_percent[valid] = (rates_valid * 100.0).astype(np.float32)
n_effettivo[valid] = cars_valid.astype(np.float32)

# 6) Save to two separate GeoTIFF files (same grid, same CRS)
share_da = xr.DataArray(
    share_percent,
    coords=score_da.coords,
    dims=score_da.dims,
    name="share",
)
share_da.rio.write_crs(score_da.rio.crs, inplace=True)
share_da.rio.write_nodata(output_nodata, inplace=True)
share_da.attrs["long_name"] = "vehicles_allocation_share_percent"
share_da.attrs["total_vehicles_target"] = int(total_vehicles)
share_da.attrs["alpha"] = float(alpha)
share_da.rio.to_raster(output_share_path, compress="DEFLATE", nodata=output_nodata)

cars_da = xr.DataArray(
    n_effettivo,
    coords=score_da.coords,
    dims=score_da.dims,
    name="n_effettivo",
)
cars_da.rio.write_crs(score_da.rio.crs, inplace=True)
cars_da.rio.write_nodata(output_nodata, inplace=True)
cars_da.attrs["long_name"] = "vehicles_allocation_number_of_cars"
cars_da.attrs["total_vehicles_target"] = int(total_vehicles)
cars_da.attrs["alpha"] = float(alpha)
cars_da.rio.to_raster(output_cars_path, compress="DEFLATE", nodata=output_nodata)

# 7) Minimal summary of allocation result
total_allocated = float(np.sum(n_effettivo[n_effettivo != output_nodata], dtype=np.float64))
max_share = float(np.max(share_percent[share_percent != output_nodata]))

print(f"Estimated alpha: {alpha:.6f}")
print(f"Allocated vehicles total: {total_allocated:,.0f}")
print(f"Target vehicles total: {total_vehicles:,.0f}")
print(f"Max share (%): {max_share:.2f}")
print(f"Share raster written: {output_share_path}")
print(f"Cars raster written: {output_cars_path}")

Estimated alpha: 2.744250
Allocated vehicles total: 11,600,000
Target vehicles total: 11,600,000
Max share (%): 100.00
Share raster written: dataset_big/vehicles_allocation_share.tif
Cars raster written: dataset_big/vehicles_allocation_n_effettivo.tif


In [9]:
# Quick stats on car-ownership share distribution per pixel

import numpy as np
import rioxarray

share_path = "dataset_big/vehicles_allocation_share.tif"
share_da = rioxarray.open_rasterio(share_path)

nodata = share_da.rio.nodata
data = share_da.values.astype(np.float64, copy=False)

valid = np.isfinite(data)
if nodata is not None:
    valid &= data != nodata

vals = data[valid]

if vals.size == 0:
    raise ValueError("No valid values found in share raster.")

print("=== SHARE DISTRIBUTION (% auto per pixel) ===")
print(f"Valid cells: {vals.size:,}")
print(f"Min: {vals.min():.2f}%")
print(f"P10: {np.percentile(vals, 10):.2f}%")
print(f"P25: {np.percentile(vals, 25):.2f}%")
print(f"Median: {np.percentile(vals, 50):.2f}%")
print(f"Mean: {vals.mean():.2f}%")
print(f"P75: {np.percentile(vals, 75):.2f}%")
print(f"P90: {np.percentile(vals, 90):.2f}%")
print(f"P95: {np.percentile(vals, 95):.2f}%")
print(f"Max: {vals.max():.2f}%")

# Bins in percentage points
bins = np.array([0, 1, 5, 10, 20, 40, 60, 80, 95, 100.0001], dtype=np.float64)
labels = [
    "0-1%", "1-5%", "5-10%", "10-20%", "20-40%",
    "40-60%", "60-80%", "80-95%", "95-100%"
]

counts, _ = np.histogram(vals, bins=bins)
shares = counts / vals.size * 100.0

print("\n=== CELLS BY SHARE CLASS ===")
for lbl, c, s in zip(labels, counts, shares):
    print(f"{lbl:>8}: {c:>8,} cells ({s:>6.2f}%)")

# Extra: how many cells are near full adoption
above_90 = np.sum(vals >= 90)
above_50 = np.sum(vals >= 50)
print("\n=== QUICK THRESHOLDS ===")
print(f"Cells >= 50%: {above_50:,} ({above_50/vals.size*100:.2f}%)")
print(f"Cells >= 90%: {above_90:,} ({above_90/vals.size*100:.2f}%)")

=== SHARE DISTRIBUTION (% auto per pixel) ===
Valid cells: 563,795
Min: 0.00%
P10: 0.01%
P25: 0.03%
Median: 0.07%
Mean: 0.50%
P75: 0.24%
P90: 0.77%
P95: 1.62%
Max: 100.00%

=== CELLS BY SHARE CLASS ===
    0-1%:  519,175 cells ( 92.09%)
    1-5%:   35,748 cells (  6.34%)
   5-10%:    4,299 cells (  0.76%)
  10-20%:    2,162 cells (  0.38%)
  20-40%:    1,945 cells (  0.34%)
  40-60%:      332 cells (  0.06%)
  60-80%:       63 cells (  0.01%)
  80-95%:       48 cells (  0.01%)
 95-100%:       23 cells (  0.00%)

=== QUICK THRESHOLDS ===
Cells >= 50%: 280 (0.05%)
Cells >= 90%: 33 (0.01%)
